# Генератор датасета для классификации команд

Датасет в формате JSONL должен содержать обучающие данные для классификационной модели (например, BERT) в следующем формате:
```json
{
    "text": "Сделай громкость потише",
    "command": {
        "opcode": "SETTINGS_VOLUME",
        "recognizedArgs": {
            "group": "master",
            "action": "decrease"
        }
    }    
}
```

## Допустимые команды
Условные обозначения:
- Код операции (`opcode`): тип команды
- Распознаваемые аргументы (`recognizedArgs`): аргументы, распознанные с помощью классификационной модели для данного типа команды
- Контекстные аргументы (`contextArgs`): аргументы, передаваемые в веб-запросе и не зависящие от типа команды, но используемые для её исполнения

| Код операции      | Распознаваемые аргументы                                                                               | Контекстные аргументы | Описание                                                               |
|-------------------|--------------------------------------------------------------------------------------------------------|-----------------------|------------------------------------------------------------------------|
| `HINT_NEAREST`    | Нет                                                                                                    | TBA                   | Активировать подсказку к ближайшему заданию (если доступна)            |
| `HINT_TRUTHTABLE` | `operator: Union[TritUnOp, TritBinOp, NonBinOp]`                                                       | TBA                   | Активировать подсказку к ближайшему заданию (если доступна)            |
| `SETTINGS_VOLUME` | `group: Literal["master", "music", "sfx", "voice"]`, `action: Literal["increase", "decrease", "mute"]` | TBA                   | Увеличить / уменьшить / заглушить громкость для выбранной группы аудио |
| `PROGRESS_LEVEL`  | Нет                                                                                                    | TBA                   | Вывести информацию о текущем уровне игрока и целях на нём              |
| `FACT_RANDOM`     | `target: Literal["logic", "lore"]`                                                                     | TBA                   | Вывести случайный факт о троичной логике или вселенной игры            |
| `UNKNOWN`         | Нет                                                                                                    | Нет                   | Возвращается, если не удалось распознать команду                       |

In [1]:
import random
import json
import pymorphy3
import re
from typing import Any, Tuple, List, Dict, Set, Literal, Union, Optional

from app.services.hint.logic_ops import TritUnOp, TritBinOp, NonBinOp
from app.services.command.classifier.command_models import Command, CommandOpcode, SettingsVolumeRecognizedArgs, \
    FactRandomRecognizedArgs, HintTruthtableRecognizedArgs

In [2]:
DATASET_PATH = "./data/command_dataset.jsonl"

In [3]:
rng = random.Random(x=267)
WORD_REGEX = re.compile(r'\b\w+(?:-\w+)*\b')
COUNT = 8192
KNOWN_COUNT = int(COUNT * 0.8)

In [4]:
morph = pymorphy3.MorphAnalyzer()

def inflect_noun_or_adj(
    word: str,
    case: Literal['nomn', 'gent', 'gen1', 'gen2', 'datv', 'accs', 'ablt', 'loct', 'loc1', 'loc2'] = 'nomn',
    gender: Literal['masc', 'femn', 'neut', 'ms-f'] = 'masc',
    number: Literal['sing', 'plur'] = 'sing'        
) -> str:
    parsed_list = morph.parse(word)
    
    if not parsed_list:
        return word
    
    # Pick the first match (usually the most probable)
    parsed = parsed_list[0]
    
    # For adjectives/participles, gender is ignored if number is 'plur'
    target_grammemes = {case, number}
    if number == 'sing':
        target_grammemes.add(gender)
        
    inflected = parsed.inflect(target_grammemes)
    
    return inflected.word if inflected else word

print(inflect_noun_or_adj('троичная', case='ablt', number='plur'))
print(inflect_noun_or_adj('импликация', case='ablt', number='plur'))
print(inflect_noun_or_adj('задание', case='datv', gender='neut'))

троичными
импликациями
заданию


In [5]:
def generate_command_definition(
        include_unknown: bool = True,
        rng: Optional[random.Random] = None
) -> Tuple[CommandOpcode, Optional[Dict[str, Any]]]:
    _rng = rng if rng is not None else random
    
    hint_truthtable_operator = _rng.choice([*list(TritUnOp), *list(TritBinOp), *list(NonBinOp)])
    
    available_commands = [
        (CommandOpcode.HINT_NEAREST, None),
        (CommandOpcode.SETTINGS_VOLUME, {
            'group': _rng.choice(['master', 'music', 'sfx', 'voice']),
            'action': _rng.choice(['increase', 'decrease', 'mute']),
        }),
        (CommandOpcode.PROGRESS_LEVEL, None),
        (CommandOpcode.FACT_RANDOM, {
            'target': _rng.choice(['logic', 'lore'])
        }),
        (CommandOpcode.HINT_TRUTHTABLE, {
            'operator': hint_truthtable_operator.value,
            'balanced': (not isinstance(hint_truthtable_operator, NonBinOp)) and (_rng.random() < 0.3)
        })
    ]
    
    if include_unknown:
        available_commands.append((CommandOpcode.UNKNOWN, None))
    
    return _rng.choice(available_commands)

for i in range(16):
    command = generate_command_definition(rng=None, include_unknown=False)
    print(f'{command[0].name}: {command[1]}')
    # Command.validate({'opcode': command[0], 'recognizedArgs': command[1], 'contextArgs': None})

PROGRESS_LEVEL: None
PROGRESS_LEVEL: None
HINT_NEAREST: None
SETTINGS_VOLUME: {'group': 'music', 'action': 'mute'}
HINT_TRUTHTABLE: {'operator': 'Or', 'balanced': True}
PROGRESS_LEVEL: None
PROGRESS_LEVEL: None
SETTINGS_VOLUME: {'group': 'sfx', 'action': 'increase'}
SETTINGS_VOLUME: {'group': 'master', 'action': 'mute'}
FACT_RANDOM: {'target': 'logic'}
HINT_TRUTHTABLE: {'operator': 'Identity', 'balanced': False}
HINT_TRUTHTABLE: {'operator': 'ImplKleene', 'balanced': False}
PROGRESS_LEVEL: None
HINT_NEAREST: None
PROGRESS_LEVEL: None
PROGRESS_LEVEL: None


In [6]:
FILLERS_START: List[Tuple[str, int]] = [  # With percentages for weighted choices
    ('', 40),  # 40%
    ('пожалуйста', 10), ('эй', 5),  ('слушай', 5),  # 20%
    ('помощник', 8), ('ассистент', 8), ('шаттл', 3), ('корабль', 1),  # 20%
    ('компьютер', 6), ('товарищ компьютер', 2), ('бортовой компьютер', 2),  # 10%
    ('друг', 5), ('товарищ', 3), ('камрад', 2),  # 10%
]
FILLERS_END: List[Tuple[str, int]] = [  # With percentages for weighted choices
    ('', 70),  # 70%
    ('пожалуйста!', 8), ('плиз!', 4), ('будь добр.', 3),  # 15%
    ('быстро!', 8), ('немедленно.', 4), ('сейчас же!', 3)  # 15%
]

MESSAGE_TEMPLATES_FOR_OPCODE = {
    CommandOpcode.HINT_NEAREST: [
        'мне нужна подсказка {nearest_datv}',
        'нужна помощь {nearest_ablt}',
        'дай мне подсказку {nearest_datv}',
        'подсказка {nearest_datv}',
        'помоги мне {nearest_ablt}'
    ],
    CommandOpcode.HINT_TRUTHTABLE: [
        'нужна {truthtable_nomn} к {balanced_opt_prefix_datv}{operator_datv} {balanced_opt_suffix}',
        'дай мне {truthtable_accs} по {balanced_opt_prefix_datv}{operator_datv} {balanced_opt_suffix}',
        '{balanced_opt_prefix_nomn}{operator_nomn} {balanced_opt_suffix}',
    ],
    CommandOpcode.SETTINGS_VOLUME: [
        'нужно {modify_verb_infv} {audio_group_accs}',
        'нужно {modify_verb_infv} громкость {audio_group_gent}',
        '{modify_verb_impr} {audio_group_accs}',
        '{audio_group_nomn} {must} быть {modify_verb_prts}',
        'сделай {audio_group_accs} {modify_adverb}',
        '{modify_verb_impr} громкость {audio_group_gent}',
        'громкость {audio_group_gent} надо {modify_verb_infv}'
    ],
    CommandOpcode.PROGRESS_LEVEL: [
        'где я',
        '{locate_verb_impr} где я',
        'я где {locate_verb_impr}',
        '{locate_verb_impr} {my_pron_accs} {location_noun_accs}',
        'нужно {locate_verb_infv} {my_pron_accs} {location_noun_accs}',
        'в {which_adj_loct} я {level_noun_loct}',
        '{which_adj_short_nomn} {my_pron_nomn} {level_noun_nomn}',
        '{which_adj_short_nomn} {my_pron_nomn} {location_noun_nomn}',
        'куда я попал'
    ],
    CommandOpcode.FACT_RANDOM: [
        '{tell_verb_impr} мне {fact_noun_accs} про {target_full_accs}',
        '{tell_verb_impr} {fact_noun_accs} о {target_full_loct}',
        '{want_verb_face1} {know_verb_infv} про {target_full_accs}',
        '{fact_noun_accs} о {target_full_loct}',
        '{needed} {fact_noun_nomn} по теме - {target_full_nomn}'
    ],
    CommandOpcode.UNKNOWN: [
        'обстановка по кайфу',
        'как дела',
        'как себя чувствуешь',
        'как сам',
        'всё у меня отлично',
        'проверка связи',
        'как слышно',
    ]
}

MESSAGE_TEMPLATES_MISC = {
    'to_nearest_task_datv': ['', 'к {to_nearest_noun}', 'к {to_nearest_adj} {to_nearest_noun}', 'по {to_nearest_noun}'],
    'to_nearest_task_ablt': ['с {to_nearest_noun}', 'с {to_nearest_adj} {to_nearest_noun}']
}

MISC_WORDS = {
    'task': {
        'femn': ['задача', 'задачка', 'головоломка', 'цепь'],
        'neut': ['задание', 'уравнение'],
    },
    'nearest': ['ближайший', 'близкий', 'текущий', 'этот'],
    'truthtable': {
        'nomn': ['таблица', 'таблица истинности', 'таблица значений', 'табличка истинности', 'шпаргалка', 'подсказка'],
        'accs': ['таблицу', 'таблицу истинности', 'таблицу значений', 'табличку истинности', 'шпаргалку', 'подсказку'],
    },
    'audio_decrease_verb': ['уменьшить', 'снизить', 'понизить', 'ослабить'],
    'audio_increase_verb': ['увеличить', 'прибавить', 'повысить', 'усилить'],
    'audio_mute_verb': ['глушить', 'заглушить', 'выключить', 'отключить'],
    'audio_decrease_adverb': ['потише', 'тише', 'тихо', 'послабее'],
    'audio_increase_adverb': ['погромче', 'громче', 'громко', 'посильнее'],
    'audio_mute_adverb': ['беззвучно', 'в ноль'],
    'audio_master_group_nomn_accs': {
        'neut': ['аудио'], 'masc': ['звук', 'весь звук', 'общий звук']
    },
    'audio_music_group_noun': ['музыка', 'музычка', 'мелодия'],
    'audio_sfx_group_both': ['звуковые эффекты', 'шумовые эффекты', 'аудиоэффекты'],
    'audio_voice_group_noun': {
        'masc': ['голос', 'разговор'],
        'femn': ['речь'],
        'plur': ['речи', 'голоса', 'разговоры']
    },
    'audio_master_group_gent': ['общего звука', 'всего звука'],
    'audio_music_group_gent': ['музыки', 'мелодий'],
    'audio_sfx_group_gent': ['аудиоэффектов', 'звуковых эффектов'],
    'audio_voice_group_gent': ['голосов', 'речи', 'разговоров'],
    'must': {
        'masc': 'должен', 'femn': 'должна', 'neut': 'должно', 'plur': 'должны'
    },
    'needed': {
        'masc': 'нужен', 'femn': 'нужна', 'neut': 'нужно', 'plur': 'нужны'
    },
    'locate_verb': ['найти', 'определить', 'указать', 'выяснить'],
    'location_noun': {
        'neut': ['расположение', 'местоположение', 'местонахождение'],
        'femn': ['локация'],
        'masc': ['уровень']
    },
    'level_noun': {
        'masc': ['уровень', 'отсек', 'сектор', 'зал'],
        'femn': ['комната'],
        'neut': ['отделение']
    },
    'which_adj': {
        'loct': {
            'masc': 'каком', 'femn': 'какой', 'neut': 'каком', 'plur': 'каких'
        },
    },
    'which_adj_short_nomn': {
        'masc': 'каков', 'femn': 'какова', 'neut': 'каково', 'plur': 'каковы'
    },
    'my_pron': {
        'nomn': { 'masc': 'мой', 'femn': 'моя', 'neut': 'моё', 'plur': 'мои' },
        'accs': { 'masc': 'мой', 'femn': 'мою', 'neut': 'моё', 'plur': 'мои' },
    },
    'information_noun': {
        'masc': ['факт'],
        'femn': ['информация'],
        'neut': ['знание'],
        'plur': ['сведения', 'знания', 'факты'],
    },
    'target_logic_full': {
        'nomn': [
            'троичная логика', 'логика', 'арифметика', 'нонарная арифметика', 'девятеричная арифметика', 'тернарная логика', 'троичные вычисления', 'троичные схемы', 'схемотехника', 'нонарная система',
        ],
        'accs': [
            'троичную логику', 'логику', 'арифметику', 'нонарную арифметику', 'девятеричную арифметику', 'тернарную логику', 'троичные вычисления', 'троичные схемы', 'схемотехнику', 'нонарную систему',
        ],
        'loct': [
            'троичной логике', 'логике', 'арифметике', 'нонарной арифметике', 'девятеричной арифметике', 'тернарной логике', 'троичных вычислениях', 'троичных схемах', 'схемотехнике', 'нонарной системе',
        ],
    },
    'target_lore_full': {
        'nomn': [
            'события', 'случившееся', 'события на станции', 'состояние станции', 'произошедшее', 'история', 'хроники', 'предыстория', 'космическая программа', 'Пульсар', 'Гиперион', 'экспедиция'
        ],
        'accs': [
            'подоплёку событий', 'историю', 'хроники', 'предысторию', 'космическую программу', 'Пульсара', 'Гипериона'
        ],
        'loct': [
            'событиях', 'случившемся', 'событиях станции', 'состоянии станции', 'произошедшем', 'истории', 'хрониках', 'предыстории', 'Пульсаре', 'Гиперионе', 'экспедиции'
        ],
    },
    'want_verb_face1': ['хочу', 'желаю', 'хотел бы'],
    'know_verb_infv': ['знать', 'знать всё', 'знать больше', 'понимать всё', 'узнать'],
    'tell_verb_impr': ['расскажи', 'скажи', 'объясни'],
    'balanced_prefix_opt': {
        'nomn': {
            'femn': ['сбалансированная', 'балансная', 'симметричная'],
            'masc': ['сбалансированный', 'балансный', 'симметричный'], 'neut': ['сбалансированное', 'балансное', 'симметричное'],
        },
        'datv': {
            'femn': ['сбалансированной', 'балансной', 'симметричной'],
            'masc': ['сбалансированному', 'балансному', 'симметричному'], 'neut': ['сбалансированному', 'балансному', 'симметричному'],
        }
    },
    'unbalanced_prefix_opt': {
        'nomn': {
            'femn': ['несимметричная'], 'masc': ['несимметричный'], 'neut': ['несимметричное'],
        },
        'datv': {
            'femn': ['несимметричной'], 'masc': ['несимметричному'], 'neut': ['несимметричному'],
        }
    },
    'balanced_suffix': ['в сбалансированной форме', 'в симметричной системе', 'с балансом'],
    'unbalanced_suffix': ['в несбалансированной форме', 'в несимметричной системе', 'без баланса'],
    'operator_word': {
        'masc': { 'nomn': ['оператор'], 'datv': ['оператору'] },
        'femn': { 'nomn': ['операция', 'функция'], 'datv': ['операции', 'функции'] }
    },
    'operators': {
        'Identity': {
            'neut': { 'nomn': ['тождество'], 'gent': ['тождества'], 'datv': ['тождеству'] }
        },
        'Not': {
            'neut': { 
                'nomn': ['отрицание', 'логическое НЕ'], 'gent': ['отрицания', 'логического НЕ'], 'datv': ['отрицанию', 'логическому НЕ'] 
            },
            'femn': {
                'nomn': ['инверсия'], 'gent': ['инверсии'], 'datv': ['инверсии']
            }
        },
        'And': {
            'masc': {
                'nomn': ['энд'], 'gent': ['энда'], 'datv': ['энду']
            },
            'neut': {
                'nomn': ['логическое И', 'умножение'], 'gent': ['логического И', 'умножения'], 'datv': ['логическому И', 'умножению']
            },
            'femn': {
                'nomn': ['конъюнкция'], 'gent': ['конъюнкции'], 'datv': ['конъюнкции']
            }
        },
        'Or': {
            'neut': {
                'nomn': ['логическое ИЛИ', 'объединение'],
                'gent': ['логического ИЛИ', 'объединения'],
                'datv': ['логическому ИЛИ', 'объединению']
            },
            'femn': {
                'nomn': ['дизъюнкция'], 'gent': ['дизъюнкции'], 'datv': ['дизъюнкции']
            },
        },
        'Xor': {
            'femn': {
                'nomn': ['неравнозначность', 'ксора'], 'gent': ['неравнозначности', 'ксоры'], 'datv': ['неравнозначности', 'ксоре']
            },
            'neut': {
                'nomn': ['исключающее ИЛИ'], 'gent': ['исключающего ИЛИ'], 'datv': ['исключающему ИЛИ']
            },
            'masc': {
                'nomn': ['хор', 'ксор', 'иксор'], 'gent': ['хора', 'ксора', 'иксора'], 'datv': ['хору', 'ксору', 'иксору']
            },
        },
        'ImplLukasiewicz': {
            'femn': {
                'nomn': ['импликация Лукасевича', 'импликация', 'стрелочка', 'стрелка', 'стрелочка Лукасевича'],
                'gent': ['импликации Лукасевича', 'импликации', 'стрелочки', 'стрелки', 'стрелочки Лукасевича'],
                'datv': ['импликации Лукасевича', 'импликации', 'стрелочке', 'стрелке', 'стрелочке Лукасевича'],
            },
            'masc': {
                'nomn': ['Лукасевич', 'Лукашевич', 'Лукас'],
                'gent': ['Лукасевича', 'Лукашевича', 'Лукаса'],
                'datv': ['Лукасевичу', 'Лукашевичу', 'Лукасу'],
            }
        },
        'ImplKleene': {
            'femn': {
                'nomn': ['импликация Клини', 'стрелочка Клини', 'стрелка Клина'],
                'gent': ['импликации Клини', 'стрелочки Клини', 'стрелки Клини'],
                'datv': ['импликации Клини', 'стрелочке Клини', 'стрелке Клини']
            },
            'masc': {
                'nomn': ['Клин', 'Клини'], 'gent': ['Клина', 'Клини'], 'datv': ['Клину', 'Клини'],
            }
        },
        'NonaryPlus': {
            'neut': {
                'nomn': ['сложение', 'нонарное сложение', 'девятеричное сложение'],
                'gent': ['сложения', 'нонарного сложения', 'девятеричного сложения'],
                'datv': ['сложению', 'нонарному сложению', 'девятеричному сложению']
            },
            'masc': {
                'nomn': ['плюс', 'нонарный плюс'],
                'gent': ['плюса', 'нонарного плюса'],
                'datv': ['плюсу', 'нонарному плюсу'],
            },
        },
        'NonaryMinus': {
            'neut': {
                'nomn': ['вычитание', 'нонарное вычитание', 'девятеричное вычитание'],
                'gent': ['вычитания', 'нонарного вычитания', 'девятеричного вычитания'],
                'datv': ['вычитанию', 'нонарному вычитанию', 'девятеричному вычитанию']
            },
            'masc': {
                'nomn': ['минус', 'нонарный минус'],
                'gent': ['минуса', 'нонарного минуса'],
                'datv': ['минусу', 'нонарному минусу'],
            },
        },
        'NonaryConcat': {
            'femn': {
                'nomn': ['конкатенация'], 'gent': ['конкатенации'], 'datv': ['конкатенации']
            }
        }
    }
}

In [7]:
def mutate_text(text: str, rng: Optional[random.Random] = None) -> str:
    _rng = rng if rng is not None else random
    
    # substitute letters
    letter_replacement = {
        'ы': ['и', 'у', 'э'], 'у': ['о', 'ю'], 'а': ['я', 'э'], 'о': ['ё', 'у'], 'ъ': ['ь'], 'ь': ['ъ'], ' ': ['', ' ']
    }
    text = ''.join([
        _rng.choice(letter_replacement.get(c, [c])) if _rng.random() < 0.05 else c for c in text 
    ])
    
    # insert words between words
    stopwords = ['ну', 'эм', 'эмм', 'как бы', 'мда', 'значит', 'типа']
    words = [s for s in re.split(r'\s+', text) if s]
    if _rng.random() < 0.25:
        words.insert(_rng.randint(0, len(words) - 1), _rng.choice(stopwords))
    text = ' '.join(words)
    
    # insert up to 5 random spaces
    length = len(text)
    chars = [c for c in text]
    for i in range(_rng.randint(0, 5)):
        if _rng.random() < 0.05:
            pos = _rng.randint(0, length)
            chars.insert(pos, ' ')
            
    text = ''.join(chars).strip()
    return text


for i in range(10):
    text = "Пожалуйста, расскажи мне факты про троичные вычисления, друг."
    print(mutate_text(text, rng))

Пожалойста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, значит друг.
Пожалуйста, расскажи мне факты про троичные вычисления, дрог.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты как бы про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про ну троичные вычисления, друг.
Пожалуйста, расскажи мне факты про труичные вычисления, друг.


In [8]:
def generate_command_text_body(
    opcode: CommandOpcode,
    recognized_args: Optional[Dict[str, Any]] = None,
    rng: Optional[random.Random] = None,     
) -> str:
    _rng = rng if rng is not None else random
    
    match opcode:
        case CommandOpcode.HINT_NEAREST:
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.HINT_NEAREST])
            
            to_nearest_case = _rng.choice(['datv', 'ablt'])
            
            to_nearest_gender = _rng.choice(list(MISC_WORDS['task'].keys()))
            to_nearest_adj_ablt = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['nearest']),
                case='ablt',
                gender=to_nearest_gender
            )
            to_nearest_noun_ablt = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['task'][to_nearest_gender]), 
                case='ablt',
                gender=to_nearest_gender
            )
            to_nearest_adj_datv = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['nearest']),
                case='datv',
                gender=to_nearest_gender
            )
            to_nearest_noun_datv = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['task'][to_nearest_gender]), 
                case='datv',
                gender=to_nearest_gender
            )
            nearest_datv = _rng.choice(MESSAGE_TEMPLATES_MISC[f'to_nearest_task_datv']).format(
                to_nearest_adj=to_nearest_adj_datv,
                to_nearest_noun=to_nearest_noun_datv
            )
            nearest_ablt = _rng.choice(MESSAGE_TEMPLATES_MISC[f'to_nearest_task_ablt']).format(
                to_nearest_adj=to_nearest_adj_ablt,
                to_nearest_noun=to_nearest_noun_ablt
            )
            
            text = template.format(
                nearest_datv=nearest_datv,
                nearest_ablt=nearest_ablt
            ).strip()
            return text
        
        case CommandOpcode.SETTINGS_VOLUME:
            args = SettingsVolumeRecognizedArgs.model_validate(recognized_args)
            
            match args.group:
                case 'master':
                    gender = _rng.choice(list(MISC_WORDS['audio_master_group_nomn_accs'].keys()))
                case 'music':
                    gender = "femn"
                case 'sfx':
                    gender = "plur"
                case 'voice':
                    gender = _rng.choice(list(MISC_WORDS['audio_voice_group_noun'].keys()) + ['plur'])
                case _:
                    gender = 'plur'
            
            verb = _rng.choice(MISC_WORDS[f"audio_{args.action}_verb"])
            parsed_verb_list = morph.parse(verb)
            if parsed_verb_list:
                parsed_verb = parsed_verb_list[0]
                verb_infv = parsed_verb.inflect({ 'INFN' }).word
                verb_impr = parsed_verb.inflect({ 'impr', 'excl', ('sing' if _rng.random() < 0.5 else 'plur') }).word
                verb_prts = parsed_verb.inflect({ 'PRTS', 'pssv', 'past', gender }).word
            else:
                verb_infv = verb
                verb_impr = verb
                verb_prts = verb
                
            adverb = _rng.choice(MISC_WORDS[f"audio_{args.action}_adverb"])
                
            must_word = MISC_WORDS['must'][gender]
            
            match args.group: 
                case 'master':
                    audio_group_nomn = _rng.choice(MISC_WORDS['audio_master_group_nomn_accs'][gender])
                    audio_group_acc = audio_group_nomn
                case 'music':
                    word = _rng.choice(MISC_WORDS['audio_music_group_noun'])
                    parsed_list = morph.parse(word)
                    parsed = parsed_list[0]
                    audio_group_nomn = word
                    audio_group_acc = parsed.inflect({ 'accs' }).word
                case 'voice':
                    word = _rng.choice(MISC_WORDS['audio_voice_group_noun'][gender])
                    parsed_list = morph.parse(word)
                    parsed = parsed_list[0]
                    audio_group_nomn = word
                    audio_group_acc = parsed.inflect({ 'accs' }).word
                case 'sfx':
                    words = _rng.choice(MISC_WORDS['audio_sfx_group_both'])
                    audio_group_nomn = words
                    audio_group_acc = words
                case _:
                    audio_group_nomn = '[...]'
                    audio_group_acc = '[...]'
            
            audio_group_gent = _rng.choice(MISC_WORDS[f'audio_{args.group}_group_gent'])
                
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.SETTINGS_VOLUME])
            text = template.format(
                modify_verb_infv=verb_infv,
                modify_verb_impr=verb_impr,
                modify_verb_prts=verb_prts,
                modify_adverb=adverb,
                audio_group_nomn=audio_group_nomn,
                audio_group_accs=audio_group_acc,
                audio_group_gent=audio_group_gent,
                must=must_word
            )
            return text
        
        case CommandOpcode.PROGRESS_LEVEL:
            parsed_verb = morph.parse(_rng.choice(MISC_WORDS['locate_verb']))[0]
            verb_infv = parsed_verb.inflect({ 'INFN' }).word
            verb_impr = parsed_verb.inflect({ 'impr', 'excl', ('sing' if _rng.random() < 0.5 else 'plur') }).word
            
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.PROGRESS_LEVEL])
            
            gender = (
                _rng.choice(list(MISC_WORDS['location_noun'].keys()))
                if 'location' in template
                else _rng.choice(list(MISC_WORDS['level_noun'].keys()))
            )
            
            my_pron_nomn = MISC_WORDS['my_pron']['nomn'][gender]
            my_pron_accs = MISC_WORDS['my_pron']['accs'][gender]
            
            parsed_location_noun = morph.parse(_rng.choice(MISC_WORDS['location_noun'][gender]))[0]
            parsed_level_noun = morph.parse(_rng.choice(MISC_WORDS['level_noun'][gender]))[0]
            location_noun_nomn = parsed_location_noun.inflect({ 'nomn', gender }).word
            location_noun_accs = parsed_location_noun.inflect({ 'accs', gender }).word
            level_noun_nomn = parsed_level_noun.inflect({ 'nomn', gender }).word
            level_noun_loct = parsed_level_noun.inflect({ 'loct', gender }).word
            which_adj_loct = MISC_WORDS['which_adj']['loct'][gender]
            which_adj_short_nomn = MISC_WORDS['which_adj_short_nomn'][gender]
            
            text = template.format(
                locate_verb_infv=verb_infv,
                locate_verb_impr=verb_impr,
                level_noun_nomn=level_noun_nomn,
                level_noun_loct=level_noun_loct,
                my_pron_nomn=my_pron_nomn,
                my_pron_accs=my_pron_accs,
                location_noun_nomn=location_noun_nomn,
                location_noun_accs=location_noun_accs,
                which_adj_loct=which_adj_loct,
                which_adj_short_nomn=which_adj_short_nomn
            )
            return text
        
        case CommandOpcode.FACT_RANDOM:
            # CommandOpcode.FACT_RANDOM: [
            #     '{tell_verb_impr} мне {fact_noun_accs} про {target_full_accs}',
            #     '{tell_verb_impr} {fact_noun_accs} о {target_full_loct}',
            #     '{want_verb_face1} {know_verb_infv} больше про {target_full_accs}',
            #     '{fact_noun_accs} о {target_full_loct}',
            #     '{needed} {fact_noun_nomn} по теме - {target_full_nomn}'
            # ],
            args = FactRandomRecognizedArgs.model_validate(recognized_args)
            
            target_full_nomn = _rng.choice(MISC_WORDS[f'target_{args.target}_full']['nomn'])
            target_full_accs = _rng.choice(MISC_WORDS[f'target_{args.target}_full']['accs'])
            target_full_loct = _rng.choice(MISC_WORDS[f'target_{args.target}_full']['loct'])
            
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.FACT_RANDOM])
            
            gender = _rng.choice(list(MISC_WORDS['information_noun'].keys()))
            needed_verb = MISC_WORDS['needed'][gender]
            
            fact_noun = _rng.choice(MISC_WORDS['information_noun'][gender])
            fact_parsed = morph.parse(fact_noun)[0]
            fact_noun_nomn = fact_noun
            fact_noun_accs = fact_parsed.inflect({ 'accs', gender }).word
            
            text = template.format(
                tell_verb_impr=_rng.choice(MISC_WORDS['tell_verb_impr']),
                fact_noun_nomn=fact_noun_nomn,
                fact_noun_accs=fact_noun_accs,
                know_verb_infv=_rng.choice(MISC_WORDS['know_verb_infv']),
                target_full_nomn=target_full_nomn,
                target_full_accs=target_full_accs,
                target_full_loct=target_full_loct,
                needed=needed_verb,
                want_verb_face1=_rng.choice(MISC_WORDS['want_verb_face1']),
            )
            return text
        
        case CommandOpcode.HINT_TRUTHTABLE:
            # CommandOpcode.HINT_TRUTHTABLE: [
            #     'нужна {truthtable_nomn} к {balanced_opt_prefix_datv}{operator_datv}{balanced_opt_suffix}',
            #     'дай мне {truthtable_accs} по {balanced_opt_prefix_datv}{operator_datv}{balanced_opt_suffix}',
            #     '{balanced_opt_prefix_nomn}{operator_nomn}{balanced_opt_suffix}',
            # ],
            args = HintTruthtableRecognizedArgs.model_validate(recognized_args)
            is_balanced_prefix = _rng.random() < 0.5
            is_balanced_not_empty = (args.balanced or (_rng.random() < 0.25)) and not isinstance(args.operator, NonBinOp)
            is_operator_word = _rng.random() < 0.5
            
            operator_word_gender = _rng.choice(list(MISC_WORDS['operator_word'].keys()))
            operator_def_gender = _rng.choice(list(MISC_WORDS['operators'][args.operator.value].keys()))
            operator_gender = operator_word_gender if is_operator_word else operator_def_gender
            
            operator_word_nomn = _rng.choice(MISC_WORDS['operator_word'][operator_word_gender]['nomn'])
            operator_word_datv = _rng.choice(MISC_WORDS['operator_word'][operator_word_gender]['datv'])
            
            operator_def_nomn = _rng.choice(MISC_WORDS['operators'][args.operator.value][operator_def_gender]['nomn'])
            operator_def_gent = _rng.choice(MISC_WORDS['operators'][args.operator.value][operator_def_gender]['gent'])
            operator_def_datv = _rng.choice(MISC_WORDS['operators'][args.operator.value][operator_def_gender]['datv'])
            
            operator_nomn = f"{operator_word_nomn} {operator_def_gent}" if is_operator_word else operator_def_nomn
            operator_datv = f"{operator_word_datv} {operator_def_gent}" if is_operator_word else operator_def_datv
            
            if is_balanced_not_empty:
                _balanced_opt_prefix_nomn = _rng.choice(
                    MISC_WORDS['balanced_prefix_opt' if args.balanced else 'unbalanced_prefix_opt']["nomn"][operator_gender]
                )
                _balanced_opt_prefix_datv = _rng.choice(
                    MISC_WORDS['balanced_prefix_opt' if args.balanced else 'unbalanced_prefix_opt']["datv"][operator_gender]
                )
                _balanced_opt_suffix = (
                    _rng.choice(MISC_WORDS['balanced_suffix']
                    if args.balanced
                    else MISC_WORDS['unbalanced_suffix'])
                )
                
                balanced_opt_prefix_nomn = _balanced_opt_prefix_nomn + ' ' if is_balanced_prefix else ""
                balanced_opt_prefix_datv = _balanced_opt_prefix_datv + ' ' if is_balanced_prefix else ""
                balanced_opt_suffix = _balanced_opt_suffix if (not is_balanced_prefix) else ""
            else:
                balanced_opt_prefix_nomn = ""
                balanced_opt_prefix_datv = ""
                balanced_opt_suffix = ""
            
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.HINT_TRUTHTABLE])
            text = template.format(
                truthtable_nomn=_rng.choice(MISC_WORDS['truthtable']['nomn']),
                truthtable_accs=_rng.choice(MISC_WORDS['truthtable']['accs']),
                balanced_opt_prefix_nomn=balanced_opt_prefix_nomn,
                balanced_opt_prefix_datv= balanced_opt_prefix_datv,
                balanced_opt_suffix=balanced_opt_suffix,
                operator_nomn=operator_nomn,
                operator_datv=operator_datv,
            ).strip()
            return text
        
        case _:
            return _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.UNKNOWN])

In [9]:
def generate_command_text(
    opcode: CommandOpcode,
    recognized_args: Optional[Dict[str, Any]] = None,
    body_text: Optional[str] = None,
    mutate: bool = False,
    rng: Optional[random.Random] = None,
) -> str:
    _rng = rng if rng is not None else random
    
    def unpair(list_pairs: List[Tuple[Any, Any]]) -> Tuple[List[Any], List[Any]]:
        return [x[0] for x in list_pairs], [x[1] for x in list_pairs]
    
    text = ""
    
    prefix_options, prefix_weights = unpair(FILLERS_START)
    suffix_options, suffix_weights = unpair(FILLERS_END)
    
    prefix = _rng.choices(prefix_options, prefix_weights, k=1)[0]
    suffix = _rng.choices(suffix_options, suffix_weights, k=1)[0]
    
    if prefix:
        prefix += ", "
    if suffix:
        suffix = ", " + suffix
      
    if body_text is None:
        body = generate_command_text_body(opcode, recognized_args, rng)
    else:
        body = body_text
    
    text = prefix + body + suffix
    
    if mutate:
        text = mutate_text(text, _rng)
    
    return text
    
for i in range(32):
    # opcode = rng.choice(list(CommandOpcode))
    opcode, args = generate_command_definition(include_unknown=False, rng=rng)
    body_text = generate_command_text_body(opcode, recognized_args=args, rng=rng)
    text = generate_command_text(opcode, recognized_args=args, body_text=body_text, rng=rng)
    print(f"{opcode} () -> {text}")

CommandOpcode.SETTINGS_VOLUME () -> сделай разговоры в ноль, пожалуйста!
CommandOpcode.SETTINGS_VOLUME () -> ослабь разговоры
CommandOpcode.HINT_TRUTHTABLE () -> шаттл, нужна таблица истинности к балансному энду
CommandOpcode.HINT_NEAREST () -> бортовой компьютер, мне нужна подсказка по уравнению
CommandOpcode.HINT_NEAREST () -> помоги мне с задачей, будь добр.
CommandOpcode.HINT_TRUTHTABLE () -> эй, нужна таблица истинности к оператору тождества
CommandOpcode.FACT_RANDOM () -> пожалуйста, скажи знание о нонарной системе
CommandOpcode.HINT_TRUTHTABLE () -> нужна шпаргалка к балансной неравнозначности
CommandOpcode.PROGRESS_LEVEL () -> шаттл, какова моя комната
CommandOpcode.FACT_RANDOM () -> помощник, объясни мне факт про предысторию
CommandOpcode.PROGRESS_LEVEL () -> я где найдите
CommandOpcode.PROGRESS_LEVEL () -> ассистент, каков мой уровень
CommandOpcode.FACT_RANDOM () -> скажи мне знания про нонарную систему
CommandOpcode.SETTINGS_VOLUME () -> громкость звуковых эффектов надо усил

In [10]:
for i in range(32):
    # opcode = rng.choice(list(CommandOpcode))
    opcode, args = (CommandOpcode.HINT_TRUTHTABLE, {'operator': 'Xor', 'balanced': False})
    text = generate_command_text(opcode, recognized_args=args, rng=rng)
    print(text)

пожалуйста, нужна табличка истинности к неравнозначности без баланса, пожалуйста!
помощник, нужна таблица истинности к функции неравнозначности
пожалуйста, нужна табличка истинности к оператору неравнозначности
дай мне таблицу истинности по функции исключающего ИЛИ, будь добр.
бортовой компьютер, дай мне табличку истинности по ксоре, быстро!
нужна таблица значений к операции ксоры
помощник, дай мне таблицу истинности по операции исключающего ИЛИ, пожалуйста!
функция неравнозначности
эй, нужна таблица к функции неравнозначности
ассистент, операция исключающего ИЛИ, плиз!
помощник, ксора, быстро!
компьютер, дай мне таблицу по хору
дай мне таблицу значений по операции исключающего ИЛИ без баланса
пожалуйста, исключающее ИЛИ
ксор, быстро!
эй, нужна подсказка к операции исключающего ИЛИ, плиз!
помощник, нужна таблица истинности к оператору неравнозначности, плиз!
дай мне таблицу по несимметричному исключающему ИЛИ
товарищ, ксор
пожалуйста, дай мне табличку истинности по оператору неравнозна

In [11]:
file = open(DATASET_PATH, "a+", encoding='utf8')

for i in range(COUNT):
    opcode, args = generate_command_definition(include_unknown=False, rng=rng)
    body_text = generate_command_text_body(opcode, recognized_args=args, rng=rng)
    text = generate_command_text(
        opcode,
        recognized_args=args,
        body_text=body_text,
        mutate=(rng.random() < 0.1),
        rng=rng
    )

    obj = { 'text': text, 'command': { 'opcode': opcode.name, 'recognizedArgs': args }}
    json.dump(fp=file, obj=obj, ensure_ascii=False)
    file.write('\n')

file.close()